# 03 — Integrate Porto parish outputs and calculate walking sensitivity

This notebook joins the seven parish-level building and OD outputs into the municipal analytical universe used in the manuscript.

It performs four tasks:

1. concatenate and validate the seven building-level accessibility datasets;
2. concatenate the seven 800 m origin–destination datasets;
3. calculate the six walking scenarios at 10, 15 and 20 minutes for the **complete municipal building universe**;
4. export the final accessibility file consumed by the PCA/UAI/AAVI and spatial-aggregation notebook.

The 10-, 15- and 20-minute thresholds are used **only for sensitivity analysis**. The main accessibility specification remains the fixed 800 m network catchment.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def resolve_repo_root():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
        return cwd
    for parent in cwd.parents:
        if (parent / 'notebooks').exists() and (parent / 'data').exists():
            return parent
    raise RuntimeError('Repository root not found. Run the notebook from the repository root or from the notebooks directory.')
REPO_ROOT = resolve_repo_root()
INTERMEDIATE_ROOT = REPO_ROOT / 'data' / 'intermediate' / 'parish_accessibility'
PROCESSED_DATA = REPO_ROOT / 'data' / 'processed'
RESULTS_TABLES = REPO_ROOT / 'results' / 'tables'
RESULTS_FIGURES = REPO_ROOT / 'results' / 'figures'
for directory in [PROCESSED_DATA, RESULTS_TABLES, RESULTS_FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)
FINAL_ACCESS_FILE = PROCESSED_DATA / 'porto_building_accessibility_final.csv'
FINAL_OD_FILE = PROCESSED_DATA / 'porto_origin_destination_800m.csv'
SENSITIVITY_FILE = RESULTS_TABLES / 'walking_sensitivity_10_15_20min.csv'
MANUSCRIPT_TABLE_FILE = RESULTS_TABLES / 'Table_2_walking_sensitivity.csv'
EXPECTED_PARISH_OUTPUTS = 7
EXPECTED_BUILDINGS = 31664
EXPECTED_QC_TOBLER_CASES = 4
STRICT_MANUSCRIPT_VALIDATION = True
SPEEDS = {'0p5': 0.5, '0p7': 0.7, '0p9': 0.9}
THRESHOLDS_MIN = [10, 15, 20]
SCENARIOS = ['flat_0p5', 'flat_0p7', 'flat_0p9', 'tobler_0p5', 'tobler_0p7', 'tobler_0p9']

def source_slug(path, prefix):
    name = Path(path).stem
    name = re.sub(f'^{re.escape(prefix)}', '', name)
    name = re.sub('_global$', '', name)
    return name
print('Repository root:', REPO_ROOT)
print('Parish intermediate root:', INTERMEDIATE_ROOT)

## 1. Concatenate the seven parish building-level outputs

The final universe must contain one record per residential building. A source-parish identifier is added before concatenation so the four Tobler quality-control cases can later be repaired using parish-specific median Tobler/flat ratios.


In [ ]:
building_files = sorted(INTERMEDIATE_ROOT.rglob('resultados_acessibilidade_800m_*_global.csv'))
building_files = [p for p in building_files if 'porto_global' not in p.name.lower()]
print('Parish building files found:')
for p in building_files:
    print(' -', p.name)
if len(building_files) != EXPECTED_PARISH_OUTPUTS:
    raise FileNotFoundError(f'Expected {EXPECTED_PARISH_OUTPUTS} parish building files, found {len(building_files)}.')
parts = []
for path in building_files:
    df = pd.read_csv(path, low_memory=False)
    df['osm_id'] = df['osm_id'].astype(str).str.replace('\\.0$', '', regex=True).str.strip()
    df['ficheiro_origem'] = source_slug(path, 'resultados_acessibilidade_800m_')
    parts.append(df)
porto = pd.concat(parts, ignore_index=True, sort=False)
if porto['osm_id'].duplicated().any():
    dup = porto.loc[porto['osm_id'].duplicated(keep=False), ['osm_id', 'ficheiro_origem']]
    raise ValueError('Duplicate osm_id values were found after parish concatenation.\n' + dup.head(30).to_string(index=False))
porto['minimo_zero_snap'] = pd.to_numeric(porto.get('distancia_minima_servico'), errors='coerce').eq(0)
tobler07 = pd.to_numeric(porto['tempo_medio_seg__tobler_0p7'], errors='coerce')
porto['tobler_qc_excluded'] = tobler07.gt(60 * 60)
print('\nFinal municipal building universe:', len(porto))
print('Unique osm_id:', porto['osm_id'].nunique())
print('Zero-minimum snapping flags:', int(porto['minimo_zero_snap'].sum()))
print('Tobler mean-time QC flags:', int(porto['tobler_qc_excluded'].sum()))
if STRICT_MANUSCRIPT_VALIDATION:
    assert len(porto) == EXPECTED_BUILDINGS, f'Expected {EXPECTED_BUILDINGS} buildings, found {len(porto)}.'
    assert int(porto['tobler_qc_excluded'].sum()) == EXPECTED_QC_TOBLER_CASES, f"Expected {EXPECTED_QC_TOBLER_CASES} Tobler QC cases, found {int(porto['tobler_qc_excluded'].sum())}."

## 2. Concatenate the municipal 800 m OD table

The OD table stores network distance and unit Tobler cost for each building–destination pair within the fixed 800 m catchment. This makes the temporal sensitivity analysis independent of another network-routing run.


In [ ]:
od_files = sorted(INTERMEDIATE_ROOT.rglob('resultados_od_800m_*_global.csv'))
od_files = [p for p in od_files if 'porto_global' not in p.name.lower()]
print('Parish OD files found:')
for p in od_files:
    print(' -', p.name)
if len(od_files) != EXPECTED_PARISH_OUTPUTS:
    raise FileNotFoundError(f'Expected {EXPECTED_PARISH_OUTPUTS} parish OD files, found {len(od_files)}.')
od_parts = []
for path in od_files:
    df = pd.read_csv(path, low_memory=False)
    df['building_id'] = df['building_id'].astype(str).str.replace('\\.0$', '', regex=True).str.strip()
    df['source_parish'] = source_slug(path, 'resultados_od_800m_')
    od_parts.append(df)
od = pd.concat(od_parts, ignore_index=True, sort=False)
required_od = {'building_id', 'service_id', 'service_category', 'network_distance_m', 'tobler_unit_cost_m'}
missing_od = required_od - set(od.columns)
if missing_od:
    raise KeyError(f'Missing OD columns: {sorted(missing_od)}')
od['network_distance_m'] = pd.to_numeric(od['network_distance_m'], errors='coerce')
od['tobler_unit_cost_m'] = pd.to_numeric(od['tobler_unit_cost_m'], errors='coerce')
if od['network_distance_m'].dropna().gt(800 + 1e-06).any():
    raise AssertionError('OD pairs beyond the fixed 800 m catchment were found.')
od.to_csv(FINAL_OD_FILE, index=False)
print('Municipal OD pairs:', len(od))
print('Maximum network distance:', od['network_distance_m'].max())
print('Saved:', FINAL_OD_FILE.resolve())

## 3. Walking sensitivity at 10, 15 and 20 minutes

For each speed, flat travel time is derived from network distance and Tobler-adjusted time from the stored unit Tobler cost. Buildings with no destination within a given temporal threshold are retained with zero accessible destinations and zero accessible categories.


In [ ]:
for tag, speed in SPEEDS.items():
    od[f'time_min__flat_{tag}'] = od['network_distance_m'] / speed / 60
    od[f'time_min__tobler_{tag}'] = od['tobler_unit_cost_m'] / speed / 60
all_buildings = pd.Index(porto['osm_id'].astype(str).unique(), name='building_id')
rows = []
for threshold in THRESHOLDS_MIN:
    for scenario in SCENARIOS:
        time_col = f'time_min__{scenario}'
        reachable = od.loc[od[time_col] <= threshold].copy()
        by_building = reachable.groupby('building_id').agg(accessible_destinations=('service_id', 'count'), accessible_categories=('service_category', 'nunique')).reindex(all_buildings, fill_value=0)
        rows.append({'network_model': 'Tobler-adjusted' if scenario.startswith('tobler') else 'Flat', 'speed_m_s': SPEEDS[scenario.split('_')[1]], 'threshold_min': threshold, 'mean_accessible_destinations': by_building['accessible_destinations'].mean(), 'median_accessible_destinations': by_building['accessible_destinations'].median(), 'mean_accessible_categories': by_building['accessible_categories'].mean(), 'median_accessible_categories': by_building['accessible_categories'].median(), 'no_accessible_destinations_pct': by_building['accessible_destinations'].eq(0).mean() * 100})
sensitivity = pd.DataFrame(rows).sort_values(['network_model', 'speed_m_s', 'threshold_min']).reset_index(drop=True)
display(sensitivity.round(2))
sensitivity.to_csv(SENSITIVITY_FILE, index=False)
print('Buildings used in every sensitivity scenario:', len(all_buildings))
print('Saved:', SENSITIVITY_FILE.resolve())

## 4. Export the manuscript-form sensitivity table and Figure 5

The table is reshaped so each cell reports the 10 / 15 / 20-minute values in the same order used in the manuscript.


In [ ]:
def triplet(group, column, decimals=2):
    values = group.sort_values('threshold_min')[column].round(decimals).tolist()
    return ' / '.join((f'{v:.{decimals}f}' for v in values))
table_rows = []
for (model, speed), group in sensitivity.groupby(['network_model', 'speed_m_s'], sort=False):
    table_rows.append({'Network model': model, 'Speed (m/s)': speed, 'Mean accessible destinations (10 / 15 / 20 min)': triplet(group, 'mean_accessible_destinations'), 'Mean accessible categories (10 / 15 / 20 min)': triplet(group, 'mean_accessible_categories'), 'No accessible destinations (%) (10 / 15 / 20 min)': triplet(group, 'no_accessible_destinations_pct')})
table2 = pd.DataFrame(table_rows)
display(table2)
table2.to_csv(MANUSCRIPT_TABLE_FILE, index=False)
fig, ax = plt.subplots(figsize=(8, 5))
for (model, speed), group in sensitivity.groupby(['network_model', 'speed_m_s']):
    label = f'{model}, {speed:.1f} m/s'
    group = group.sort_values('threshold_min')
    ax.plot(group['threshold_min'], group['mean_accessible_destinations'], marker='o', label=label)
ax.set_xlabel('Travel-time threshold (min)')
ax.set_ylabel('Mean accessible destinations')
ax.set_xticks(THRESHOLDS_MIN)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / 'Figure_5_walking_sensitivity.png', dpi=600, bbox_inches='tight')
plt.show()
print('Saved:', MANUSCRIPT_TABLE_FILE.resolve())

## 5. Export the final municipal accessibility input

The exported file contains the original building-level accessibility variables, the parish source identifier and the explicit terrain-QC flags required by Notebook 04.


In [ ]:
porto.to_csv(FINAL_ACCESS_FILE, index=False)
n_no_services = int(pd.to_numeric(porto['numero_servicos_proximos'], errors='coerce').fillna(0).eq(0).sum())
mean_destinations = float(pd.to_numeric(porto['numero_servicos_proximos'], errors='coerce').mean())
service_cols = ['Supermercados', 'Bancos', 'Farmacias', 'CTT', 'Parques e jardins', 'Centro Saude', 'Hospitais']
mean_categories = float((porto[service_cols].apply(pd.to_numeric, errors='coerce').fillna(0) > 0).sum(axis=1).mean())
mean_distance = float(pd.to_numeric(porto['distancia_media_servicos'], errors='coerce').mean())
print('Final accessibility file:', FINAL_ACCESS_FILE.resolve())
print('Buildings:', len(porto))
print('No accessible destinations within 800 m:', n_no_services)
print('Mean accessible destinations:', round(mean_destinations, 2))
print('Mean functional categories:', round(mean_categories, 2))
print('Mean network distance:', round(mean_distance, 2), 'm')
if STRICT_MANUSCRIPT_VALIDATION:
    assert n_no_services == 420
    assert np.isclose(mean_destinations, 16.18, atol=0.01)
    assert np.isclose(mean_categories, 4.4, atol=0.01)
    assert np.isclose(mean_distance, 540.13, atol=0.05)
print('Municipal integration validation: OK')